![image](car.jpeg)

**Car-ing is sharing**, une concession spécialisée dans la vente et la location de voitures, fait passer ses services au niveau supérieur grâce aux **Large Language Models (LLMs)**.

En tant que nouveau développeur IA et NLP de l’équipe, vous devez créer un prototype d’application de chatbot aux fonctionnalités multiples, destiné à la fois à accompagner les clients et à soutenir les équipes internes.

La solution doit recevoir des invites textuelles et s’appuyer sur différents LLMs préentraînés de Hugging Face pour répondre à une série de tâches : par exemple, classifier le sentiment d’un avis textuel sur une voiture, répondre à une question client, résumer ou traduire un texte, etc.


In [6]:
# Import necessary packages
import pandas as pd
import torch

from transformers import logging
logging.set_verbosity(logging.WARNING)

In [9]:
# Import necessary packages
import pandas as pd
import torch
from transformers import logging
logging.set_verbosity(logging.WARNING)

# Start your code here!

# --- Chargement des données ---
df = pd.read_csv("data/car_reviews.csv", sep=";")
reviews = df["Review"].tolist()
real_labels = df["Class"].tolist()  # ex: "POSITIVE"/"NEGATIVE"

# ============================================================
# Tâche 1 : Classification de sentiment
# ============================================================
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score

classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
predicted_labels = classifier(reviews)

# Mapping des labels texte -> entiers binaires {0,1}
label_map = {"POSITIVE": 1, "NEGATIVE": 0}
predictions = [label_map[pred["label"]] for pred in predicted_labels]
references = [label_map[label.upper()] for label in real_labels]

accuracy_result = accuracy_score(references, predictions)
f1_result = f1_score(references, predictions)

print("Accuracy:", accuracy_result)
print("F1:", f1_result)

# ============================================================
# Tâche 2 : Traduction (EN -> ES) + score BLEU
# ============================================================
import evaluate

first_review = reviews[0]
# Extraction des deux premières phrases
two_sentences = ". ".join(first_review.split(". ")[:2]) + "."

translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es")
translation_output = translator(two_sentences)
translated_review = translation_output[0]["translation_text"]

with open("data/reference_translations.txt", "r") as f:
    lines = f.readlines()
references_bleu = [line.strip() for line in lines]

# IMPORTANT : utiliser evaluate.load("bleu").compute() -> renvoie un dict
bleu = evaluate.load("bleu")
bleu_score = bleu.compute(
    predictions=[translated_review],
    references=[references_bleu]
)

print("Translated:", translated_review)
print("BLEU score:", bleu_score)

# ============================================================
# Tâche 3 : Question-réponse extractive sur le 2e avis
# ============================================================
second_review = reviews[1]
context = second_review
question = "What did he like about the brand?"

qa_pipeline = pipeline(
    "question-answering",
    model="deepset/minilm-uncased-squad2",
    tokenizer="deepset/minilm-uncased-squad2"
)
qa_result = qa_pipeline(question=question, context=context)
answer = qa_result["answer"]
print("Answer:", answer)

# ============================================================
# Tâche 4 : Résumé du dernier avis
# ============================================================
last_review = reviews[-1]
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
summary_output = summarizer(last_review, max_length=55, min_length=50, do_sample=False)
summarized_text = summary_output[0]["summary_text"]
print("Summary:", summarized_text)

Device set to use cpu


Accuracy: 0.8
F1: 0.8571428571428571


Device set to use cpu
Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Translated: Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta camioneta para mis entregas de negocios y uso personal.
BLEU score: {'bleu': 0.7794483794144497, 'precisions': [0.9090909090909091, 0.8571428571428571, 0.75, 0.631578947368421], 'brevity_penalty': 1.0, 'length_ratio': 1.0476190476190477, 'translation_length': 22, 'reference_length': 21}
Answer: ride quality, reliability


Device set to use cpu


Summary: The Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. The engine delivers strong
